# Stage 4 — Phase 2: Data Preparation

Builds on Phase 0-1 (`Stage4_Phase0-1_Data_Inspection.ipynb`) — assumes datasets are present,
verified, and integrity-checked. Starts with 2.1: locking the Gupta test split.

## 0. Configuration + mount

In [ ]:
DRIVE_ROOT   = "/content/drive/MyDrive/pid_project/data"
KAGGLE_DIR   = "kaggle_pid_symbols"
GUPTA_DIR    = "gupta_pid"
PID2GRAPH_DIR= "pid2graph"
DATA_VERSION = "data-v1"

from google.colab import drive
drive.mount('/content/drive')

import os, json, hashlib
from pathlib import Path

ROOT = Path(DRIVE_ROOT)
gupta_p = ROOT / GUPTA_DIR
print("Gupta root:", gupta_p, "| exists:", gupta_p.exists())

## 2.1 Lock the Gupta test split

Per CLAUDE.md hard rule #1: the 20-sheet test split is **never opened during training or
tuning** — touched exactly once, at final scoring. This cell freezes `test_ids.json` to
Drive as the single source of truth. If the file already exists, it verifies against it
rather than silently overwriting — the whole point is that this file doesn't drift.

In [ ]:
raw = gupta_p / "PID_Dataset" / "0__raw_data"
train_ids = sorted(p.stem for p in (raw / "sheets" / "train").iterdir())
test_ids  = sorted(p.stem for p in (raw / "sheets" / "test").iterdir())

assert len(train_ids) == 72, f"expected 72 train sheets, got {len(train_ids)}"
assert len(test_ids) == 20, f"expected 20 test sheets, got {len(test_ids)}"

overlap = set(train_ids) & set(test_ids)
assert not overlap, f"train/test overlap found: {overlap}"

split_record = {
    "data_version": DATA_VERSION,
    "train_ids": train_ids,
    "test_ids": test_ids,
}

test_ids_path = ROOT / "test_ids.json"

if test_ids_path.exists():
    existing = json.loads(test_ids_path.read_text())
    assert existing["test_ids"] == test_ids, (
        "test_ids.json already exists and DIFFERS from the current split — "
        "this file is frozen per CLAUDE.md hard rule #1. Investigate before overwriting; "
        "do not just re-run and accept a new split."
    )
    print(f"test_ids.json already exists and matches current split — unchanged. ({test_ids_path})")
else:
    test_ids_path.write_text(json.dumps(split_record, indent=2))
    print(f"Wrote frozen split to {test_ids_path}")

print(f"\ntrain: {len(train_ids)} | test: {len(test_ids)} | overlap: {len(overlap)}")
print("✓ 72/20 split locked, zero overlap")